-----------
<br><br><a id="0"></a>
# 0. ***Introduction***
---------------------------------
We build a GPT, following the paper "Attention is All You Need" and OpenAI's GPT-2 / GPT-3. We talk about connections to ChatGPT, which has taken the world by storm. We watch GitHub Copilot, itself a GPT, help us write a GPT (meta :D!). We'll utilise the small "[Tiny Shakespeare](https://raw.githubusercontent.com/jcjohnson/torch-rnn/master/data/tiny-shakespeare.txt)" dataset, which contains all of Shakespeare's work in a single file under $1$ MB, instead of a bigger chunk-sized entire internet dataset. This will tremendously reduce our parameter size from the billions. For simplicity and speed, our input tokens will be characters and not words. It's essential to watch the earlier makemore videos to get comfortable with the autoregressive language modeling framework, and basics of tensors & `PyTorch`'s **`torch.nn`**, which we take for granted in this video.

**ChatGPT** is a language model (LM) developed & designed by OpenAI to understand and generate human-like text sequentially based on the input it receives. You can use it for various natural language processing tasks, such as answering questions, having conversations, generating text, and more. For the same input, it provides different outputs when it's rerun numerous times. This shows that it's a probabilistic LM.

<u>Generative Pre-trained Transformer,</u> otherwise known as **GPT**, is a LM that is trained on a siginificant large size of text data to understand and generate human-like text sequentially. The "transformer" part refers to the model's architecture, which was introduced and inspired by the 2017 "[Attention Is All You Need](https://arxiv.org/abs/1706.03762)" paper.

Current implementations from **micrograd (n-grams LM)** to **makemore (MLP, CNN, RNN)** and now **GPT** follow a few key papers:

- Bigram (one character predicts the next one with a lookup table of counts)
- MLP, following [Bengio et al. 2003](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf)
- CNN, following [DeepMind WaveNet 2016](https://arxiv.org/abs/1609.03499) (in progress...)
- RNN, following [Mikolov et al. 2010](https://www.fit.vutbr.cz/research/groups/speech/publi/2010/mikolov_interspeech2010_IS100722.pdf)
	- LSTM, following [Graves et al. 2014](https://arxiv.org/abs/1308.0850)
	- GRU, following [Kyunghyun Cho et al. 2014](https://arxiv.org/abs/1409.1259)
- Transformer, following [Vaswani et al. 2017](https://arxiv.org/abs/1706.03762)

In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
import math
import numpy as np

-----------
<br><br><a id="1"></a>
# 1. Baseline Bigram Language Model (LM)
-----------

We establish a simple bigram language model (LM) to get started as our baseline LM. We build our dataset, create our input tokens, split it into train and validation sets, create our bigram LM, train the model and then measure the model performance via cross-entropy loss.


### What is a Bigram?
A **bigram** model looks at **one character** and predicts the **next character**.
"Bi" means two — so bigram = a pair of characters.

**Example from "Hello":**
- H → e
- e → l
- l → l
- l → o

If I show you just **"H"**, can you guess the next letter?
Maybe **"e"** — because "He" is very common in English.
Now if I show you **"He"**, you're even more confident the next letter is **"l"**.
That's exactly what the bigram model does — it learns from all pairs it sees in training data.

### Weakness
The model is **completely blind** to anything before the current character.
It has no memory, no context — which is exactly why we build GPT later.


<a id="101"></a>
## 1.1. Data Reading & Exploration
-----------

Let's download the Tiny Shakespeare dataset, which is about a $1$ MB file, that contains all of Shakespeare's work in one single text file. We read in the text file and, upon inspection, discover it has ~$1$ million characters.

In [ ]:
import urllib.request

urllib.request.urlretrieve(
	'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
	'input.txt'
)

with open('input.txt', 'r', encoding='utf-8') as f:
	text = f.read()

print(f"Total characters: {len(text)}")
print()
print(text[:500])

Total characters: 1115394

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [5]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("Vocab Size :", vocab_size)
print("Unique characters in dataset :",''.join(chars))
print("\nTotal number of unique characters in dataset:", vocab_size, '\n')



Vocab Size : 65
Unique characters in dataset : 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz

Total number of unique characters in dataset: 65 



<a id="102"></a>
## 1.2.  Tokenization & Train-Dev Split
-----------


A **tokenizer** is a component used in natural language processing (NLP) to convert raw text of strings into some sequence of integers known as "<u>tokens</u>". An **encoder** allows us turn tokens represented as strings into integers, and a **decoder** allows us to turn our tokens represented as integers back into strings.

We have a very simple character-level tokenizer. There are many different tokenizers, like Google's [SentencePiece](https://github.com/google/sentencepiece) schema (**a subword tokenizer**) or OpenAI's [tiktoken](https://github.com/openai/tiktoken) (**a byte pair encoding, BPE, tokenizer**). These tokenizers operate fundamentally on a sub-word level, which means their vocabulary is much larger (since there are many more permutations of subwords than characters). But the general idea remains the same, we are just turning strings into integers and vice versa.

The large vocabulary size of <u>tiktoken</u>, which is $50257$, enables us to encode a string to a shorter sequence of integers as compared to our own tokenizer of size 65 which generates a longer sequence of integer tokens. The larger the vocabulary size, the shorter the sequence of integer tokens.

So, once we define our encoder and decoder we can then encode our entire dataset. Once we have our encoded dataset, we perform a $90\%:10\%$ train-validation split.

In [6]:
# create a mappong from characters to integers 

stoi = {ch:i for i, ch in enumerate(chars)} # string to integer
itos = {i:ch for i, ch in enumerate(chars)} # integer to string
encode = lambda s: [stoi[c] for c in s]     # encoder: take a string , output a list of integers
decode = lambda l: "".join([itos[i] for i in l])  # decode: take a list of integers, output a string 

print(encode("abdallah"))
print(decode(encode("abdallah")))
print(decode([39, 40, 42, 39, 50, 50, 39, 46]))

[39, 40, 42, 39, 50, 50, 39, 46]
abdallah
abdallah


In [7]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch 

data = torch.tensor(encode(text), dtype= torch.long)
print("size:",data.shape, "\ndtype:",data.dtype)
print(data[:500])

size: torch.Size([1115394]) 
dtype: torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 

In [8]:
# Let's now split up the data into train and validation sets
n = int(0.9 * len(data)) # first 90% will be train, remaining 10% will be val
train_data = data[:n]
val_data = data[n:]

<a id="103"></a>
## 1.3.  Data Loader: Batches
-----------

Let's prepare the model input. We will never feed our model the entire sequence of tokens as prompt at once.
Instead, we will feed it a **randomly drawn but consecutive sequence of tokens.** The model will then predict the next token in the sequence from this prompt.

>We refer to these consecutive, size-limited input sequences of tokens as ***blocks***.
Size-limited means that blocks can have a length of up to `block_size`.

When we sample our dataset, we grab a block of $8$ characters of context plus 1 final character as target. The goal is to learn from the target character during training, predict from the target during evaluation, and generate text from the target during inference.

Suppose we have a `block_size` of $8$, each block actually contains 8 different examples, one for each possible sequence starting with the $1$st initial character. It is important to show our model examples with fewer than `block_size` characters, so that it can learn how to generate text with as little as one character context. Essentially, the transformer should be robust to varying context lengths (1 to `block_size`), which is essential during inference (adequate text generation during sampling with as little as context length of 1 to `block_size`).

In [9]:
block_size = 8 
train_data[:block_size +1 ] # the first 9 char in the data set 

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [ ]:
x = train_data[:block_size]     # x = train_data[:8]   --> x = [18, 47, 56, 57, 58,  1, 15, 47]
y = train_data[1:block_size+1]  # y = train_data[1:9]  --> y = [47, 56, 57, 58,  1, 15, 47, 58]
for t in range(block_size): # range (0,1,2,3,4,5,6,7)
	context = x[:t+1]       # x[1] --> x[2] --> x[3] --> x[4] --> x[5] --> ....
	target = y[t]           # y[0] --> y[1] --> ....
	print(f"{t+1}.when input is, {context}, the target: {target}")

1.when input is, tensor([18]), the target: 47
2.when input is, tensor([18, 47]), the target: 56
3.when input is, tensor([18, 47, 56]), the target: 57
4.when input is, tensor([18, 47, 56, 57]), the target: 58
5.when input is, tensor([18, 47, 56, 57, 58]), the target: 1
6.when input is, tensor([18, 47, 56, 57, 58,  1]), the target: 15
7.when input is, tensor([18, 47, 56, 57, 58,  1, 15]), the target: 47
8.when input is, tensor([18, 47, 56, 57, 58,  1, 15, 47]), the target: 58


In [ ]:
print('X:', decode(x.tolist()), '  ||  y:', decode(y.tolist()),'\n')
for t in range(block_size):
	context = x[:t+1].tolist()
	target = y[t].tolist()
	print(f"{decode(context)} → {decode([target])}")

X: First Ci   ||  y: irst Cit 

F → i
Fi → r
Fir → s
Firs → t
First →  
First  → C
First C → i
First Ci → t


In the cell above, the representation of X and y is different from our `makemore` version. In makemore, we had a **fixed input context size,** and we padded with `.` in cases where the names were not the full context length. Here, we append each subsequent character step-by-step to ensure the LM learns robustly to **varying context lengths from 1 to `block_size`.**


Now, we feed in the dataset in **batches** of multiple chunks of text that are all stacked up like in a single tensor. This is done for efficiency and speed since GPUs are good at parallel processing/computing. The batches are processed simultaneously and independently of each other.

Since we have `batch_size` 4 and `block_size` 8, one batch will contain a $4\times8$ tensor $X$ and a $4\times8$ tensor $Y$.

* Each row, as a single sample, contains 8 different example contexts, one for each possible sequence starting with the $1$st character until the `block_size`.
* There are 4 rows for the 4 samples in a single batch of `batch_size` 4. Each row has 8 examples, therefore there's a total of 32 training samples.
* Each element in the 4x8 tensor Y contains a single target, each corresponding to one of the 32 examples in X.

In [12]:
print(len(data))

1115394


***torch.stack()*** combines multiple tensors and puts them together along a new dimension.

**Example :**
a = torch.tensor([1, 2, 3]) 
b = torch.tensor([4, 5, 6])

result = torch.stack([a, b])

**Result** : 
tensor([
	[1, 2, 3],
	[4, 5, 6]
])

In [ ]:
torch.manual_seed(1337) 
batch_size = 4  # how many independent sequence will we process in parallet ?
block_size = 8  # what is the maximum context length for predictions ?

def get_batch(split):
	# generate a small batch of data of inputs x and target y
	data = train_data if split == "train" else val_data
	ix = torch.randint(len(data) - block_size, (batch_size,))  # pick random positon between 0 and  (1115394 - 8, (4,)), assume ix = [10, 100, 500, 1000]
	x = torch.stack([data[i:i+block_size] for i in ix])        # let's assume i = 10, data[10:18], → [A B C D E F G H] || i = 100, data[100:108], → [K L M N O P Q R] ||--> x =[[A B C D E F G H],[K L M N O P Q R]]
	y = torch.stack([data[i+1:i+block_size+1] for i in ix])
	return x, y

xb, yb = get_batch('train')
print("input: ")
print("xb.shape :",xb.shape)
print("xb :",xb)
print()
print("target :")
print("yb.shape:", yb.shape)
print("yb :",yb)

print("-----")

for b in range(batch_size):   # batch dimension
	for t in range(block_size): #time dimension
		context = xb[b,:t+1]
		target = yb[b,t]
		print(f"when input is {context.tolist()} the target is : {target}")

input: 
xb.shape : torch.Size([4, 8])
xb : tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])

target :
yb.shape: torch.Size([4, 8])
yb : tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
-----
when input is [24] the target is : 43
when input is [24, 43] the target is : 58
when input is [24, 43, 58] the target is : 5
when input is [24, 43, 58, 5] the target is : 57
when input is [24, 43, 58, 5, 57] the target is : 1
when input is [24, 43, 58, 5, 57, 1] the target is : 46
when input is [24, 43, 58, 5, 57, 1, 46] the target is : 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target is : 39
when input is [44] the target is : 53
when input is [44, 53] the target is : 56
when input is [44, 53, 56] the target is : 1
when input is [44, 53, 56, 1] the targ

<a id="104"></a>
## 1.4. Bigram LM
-----------
Lets start with the simplest model possible, which is a bigram language model, a character-level language model that generates the next character based on the previous one and bases its generation on the probability of two characters occurring together. 

### Forward pass
Below we implement the bigram language model using an embedding with exactly `vocab_size x vocab_size`. Embedding a single integer between `0` and `vocab_size-1` would return a tensor of length `vocab_size`. This acts like a lookup table, where passing in a row index between `0` and `vocab_size-1` would return a row with length `vocab_size`. We simply initialize an embedding that maps each token to a probability distribution for the next token.

If we pass in a multi-demensional vector as input, the embedding simply returns a tensor with the same dimensions, excecpt each integer gets turned into a vector of `vocab_size`. For example, if we pass in an input with dimensions `BxT`, then the output will be have dimension `BxTxC`.

* `B` is the "batch" dimension, indicating which sequence of the batch we are in, equal to `batch_size`.
* `T` is the "time" dimension, indicating our position in the sequence, equal to `block_size`.
* `C` is the "channel" dimension, indicating which neuron we are talking about, equal to `vocab_size`.



Ensure you pass in `logits` and `target` with the right **shape** when calling `F.cross_entropy`. The loss we expect, given a uniform distribution, to make a prediction is: <br>
$$-ln(\frac{1}{vocab\_size})=-ln(\frac{1}{65})=4.17387$$ <br>
However, we get a higher loss of $\boldsymbol{4.8786}$ which shows that initial predictions are not super diffused or evenly spread out across the entire `vocab_size` and contain a bit of entropy.

In [14]:
float(-np.log(1/vocab_size))  # vocab_size = 65
# so we expext the loss to be around 4.17

4.174387269895637

In [ ]:
import torch 
import torch.nn as nn 
from torch.nn import functional as F 
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

	def __init__(self, vocab_size):
		super().__init__()
		# each token derectly reads off the logits for the next token from a lookup table
		self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)  # (65, 65)

	def forward(self, idx, target):   # idx = xb  || target = yb
		#idx and targets are both (B,T)  tensor of integers 
		logits = self.token_embedding_table(idx) #(B,T,C)
		return logits

m = BigramLanguageModel(vocab_size)
out = m(xb, yb)
print(out.shape)


torch.Size([4, 8, 65])


In [16]:
print(xb)
print("-------------------------------------------------")
print(yb)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
-------------------------------------------------
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


--------------------------------------

```markdown
Each token represents an input token.

For each input token, we have 65 scores,
one score for each possible next token.

For example:

idx = xb

[24, 43, 58, 5, 57, 1, 46, 43]

We take each number and use it to look up a row
from the embedding table.

For example:

token 24
	↓
[ 0.2, -0.4, 1.3, 0.1, ..., 0.7 ]
	←──────── 65 logits ────────→

These 65 numbers are the scores for the
65 possible next tokens.

They are called logits, not probabilities yet.

Later, softmax converts these logits into probabilities.
```
--------------------------------------

In [17]:
import torch 
import torch.nn as nn 
from torch.nn import functional as F 
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

	def __init__(self, vocab_size):
		super().__init__()
		# each token derectly reads off the logits for the next token from a lookup table
		self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)  # (65, 65)

	def forward(self, idx, targets = None):   # idx = xb  || target = yb
		#idx and targets are both (B,T)  tensor of integers 
		logits = self.token_embedding_table(idx) #(B,T,C)

		if targets is None:
			loss = None
		else :
			B, T, C = logits.shape
			logits = logits.view(B*T, C)
			targets = targets.view(-1)  # or you can do (B*T)
			loss = F.cross_entropy(logits, targets)
		return logits, loss

	def generate(self, idx, max_new_tokens):
		#idx is (B,T) array of indices in the current context
		for _ in range(max_new_tokens):
			#get the predictions 
			logits, loss = self(idx)
			# focus only on the last time step
			logits = logits[:, -1, :] #become (B,C)
			# apply softmax to get probabilities 
			probs = F.softmax(logits, dim=-1) # (B, C)
			# sample from the distribution
			idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
			# append sampled index to the running sequence
			idx = torch.cat((idx, idx_next), dim = 1) # (B, T+1)

		return idx

	
bigramLM = BigramLanguageModel(vocab_size)
logits, loss = bigramLM(xb, yb)
print(logits.shape)
print(loss)


torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)


### Generate Function :
Let's add the ability to generate characters to our model. To generate our output, we take the logits which would be the conditional probabilities of two characters occurring together after the last function, and extract the last token in each block because that will be the token that we will use for generating the succeeding characters. Then, we apply softmax on the last dimension which contains the output probabilities.


`generate` takes some context and uses it to generate `max_new_tokens` more characters.

For each new token up to `max_new_tokens`:

* call the forward pass with the given context `idx` (without targets) to get the logits
* "pluck out" the logits for just the last position in dimension `T` (since our forward pass acts on all `BxT` inputs and returns `BxTxC`)
* apply ***softmax*** to the last position (`BxC`) to transform into probabilities
> **Softmax** essentially amplifies the differences between the elements of the input vector, converting them into probabilities that represent the likelihood of each class or category. The softmax function transforms logits (raw scores) into probabilities that sum up to 1, and each probability represents the likelihood of a particular class. The distribution of these probabilities depends on the distribution of the logits themselves.
* sample from the probability distribution to generate the next character
* append generated character to context and "shift" the context window
* repeat

See that `self(idx)` calls the `forward` function of the model. `forward` is adapted accordingly above to also take a call with just `idx`.

In [18]:
# initial context is just a 0
idx = torch.zeros((1, 1), dtype=torch.long)

# generate 100 new tokens
res = bigramLM.generate(idx, max_new_tokens=100)

# take the first batch
res0 = res[0]

# decode
print(decode(res0.tolist()))


SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


<a id="105"></a>
## 1.5. Training Bigram LM
-----------

Let's train our model. Let's setup our optimization routine. We will use the AdamW optimizer.

* **SGD** (Stochastic Gradient Descent): A fundamental optimization algorithm used in machine learning and deep learning. It updates model parameters by computing gradients using randomly selected small batches of data, making it "stochastic." It's widely used for training neural networks and other machine learning models.

* **Adam** (Adaptive Moment Estimation): A popular optimization algorithm that improves convergence and training speed compared to traditional SGD. It maintains moving averages of gradients and adapts learning rates for each parameter. It's known for its efficiency in practice.

* **AdamW**: A modification of the Adam optimizer designed to handle weight decay (L2 regularization) more effectively. It separates weight decay from the optimization process, making it better at controlling overfitting during the training of deep neural networks. It's a preferred choice for tasks where regularization is important.

We set the learning rate to `1e-3` which is a decent setting for small networks. We estimate the loss after every $200$ steps by taking the average to prevent a noisy plot and get a more respresentative, smoother plot. We print out the estimated loss value after every $500$ steps.

In [68]:
# create a PyTorch optimizer 
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:
# create a PyTorch optimizer 
optimizer = torch.optim.AdamW(
    bigramLM.parameters(),
    lr=1e-2
)

In [72]:
batch_size = 32 
for steps in range(10000):  # increase number of steps for good results...

	# sample a batch of data
	xb, yb = get_batch('train')

	# exaluate the loss
	logits , loss = bigramLM(xb, yb)          # forward pass
	optimizer.zero_grad(set_to_none=True)     # clear accumulated gradients
	loss.backward()                           # backward pass (backprop: to get gradients)
	optimizer.step()                          # update parameters

	#print(loss.item())     # TO SEE ALL THE RESULTS 

print(loss.item())

2.6946637630462646


In [80]:
print(decode(bigramLM.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=400)[0].tolist()))


I KI:
R:JFIUE:
Con;Swith k$
Fbou manor micem$Flinsh ppr, ishen!
INThelx: bo.

BarI,Gghar whe hendit ho w, hitheeeYGMy te CEE: ththando be I'!
Cavetkzer IEYw!

Lp, hy llFat ombar, BTokirminENINor rl titl he arken CKCinWWd:ze w$XEGee tor-balantUSAIof myvefa bll den dh f my ye, r$zPlovects brst ng co hind,elllls?aus athasthimymeliNoicWhepatX&:viganERJhe frda army h;gllt, &!wne nDq,
DUSine chin's tedg


------------------------------------------

*So here we're starting to get something at least like reasonable, certainly not shakes spears but the model is making progress.
Obviosly this is a very simple model because the tokens are not talking to each other, so given the previous context of whatever was generated we're only looking at the very last character to make the predictions about what comes next.*
*Now these tokens have to start talking to each other and figuring out what is in the context so that they can make better predictions for what comes next.*


------------------------------------------

In [81]:
eval_iters = 200
max_iters = 10000
eval_interval = 500


@torch.no_grad()                           # Disable gradient calculation for this function
def estimate_loss(model):
    out = {}
    model.eval()                           # Set model to evaluation/inference mode
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()                         # Set model back to training mode
    return out

train_losses = []
val_losses = []
epochs = []

# Training
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss(bigramLM)
        train_losses.append(losses['train'].item())
        val_losses.append(losses['val'].item())
        epochs.append(iter)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = bigramLM(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long)
print(decode(bigramLM.generate(context, max_new_tokens=500)[0].tolist()))

step 0: train loss 2.6061, val loss 2.6132
step 500: train loss 2.6160, val loss 2.6136
step 1000: train loss 2.6082, val loss 2.6179
step 1500: train loss 2.5997, val loss 2.6221
step 2000: train loss 2.6049, val loss 2.6196
step 2500: train loss 2.6019, val loss 2.6214
step 3000: train loss 2.6158, val loss 2.6082
step 3500: train loss 2.6090, val loss 2.6112
step 4000: train loss 2.6115, val loss 2.6202
step 4500: train loss 2.6062, val loss 2.6233
step 5000: train loss 2.6078, val loss 2.6135
step 5500: train loss 2.6058, val loss 2.6124
step 6000: train loss 2.6002, val loss 2.6227
step 6500: train loss 2.6024, val loss 2.6195
step 7000: train loss 2.6085, val loss 2.6147
step 7500: train loss 2.6113, val loss 2.6143
step 8000: train loss 2.6121, val loss 2.6232
step 8500: train loss 2.6091, val loss 2.6147
step 9000: train loss 2.6056, val loss 2.6162
step 9500: train loss 2.6070, val loss 2.6205

A:deve3-wn: la;ALth p ZA:Anbe. herdhoutte, it$3fas!

'rthtW
Cd ape BMmm sr mitsou W

-----------
<br><br><a id="2"></a>
# 2. Self-Attention
-----------
<a id='c0'></a>
**Attention** is a communication mechanism that allows models to focus on different parts of the input data when making predictions. This concept is especially important in sequence-based NLP tasks such as machine translation and text summarization, and image processing tasks such as image captioning. It helps models pick out the important bits from a lot of information and focus on them to make smarter decisions. It overcomes the long-range dependency limitations of RNNs & LSTMs by allowing the model to weigh the importance of different elements in the sequence. Instead of processing each element sequentially, attention enables the model to look at all elements simultaneously and decide which ones are more relevant to the current task.

An attention function can be described as mapping a query and a set of key-value pairs to an output, where the **query, keys, values**, and output are all vectors. The output is computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key.

Overall, attention is good for capturing long-range dependencies, parallel processing, and making model decisions more interpretable.



<a id="201"></a>
## 2.1. V1: Averaging Past Context with `For` Loops - Weakest Form of Aggregation
-----------
We want our tokens to talk to each other. Tokens must talk only with the previous tokens. Since we are predicting the next token, we need to consider the previous tokens only (5th token communicates with 1st, 2nd, 3rd & 4th tokens)

The easiset way to make them communicate is by averaging the previous tokens embeddings. This is a weak form of interaction, it is extremely lossy since we are losing the spatial information of the token arrangements and positions.

In [82]:
# Consider the following toy example

torch.manual_seed(1337)
B,T,C = 4,8,2         # Batch, Time, Channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2                               # batch, time (tokens or block_size), channels (vocab_size)
x = torch.randn(B,T,C)
print("x:", x.shape, "\n")

# We want x[b, t] = mean_{i <= t} x[b, i]
xbow = torch.zeros((B, T, C))               # Create tensor of zeros of shape (B, T, C) (bag of words representation of the input)
for b in range(B):                          # For all batches
    for t in range(T):                      # For all tokens in the batch
        xprev = x[b, :t+1]                  # Get all tokens up to and including the current token (t, C)
        xbow[b, t] = torch.mean(xprev, 0)   # Calculate the mean of the tokens up to and including the current token

print('Batch [0]:\n', x[0], "\n")     # First batch of 8 tokens, each of size 2
print('Running Averages:\n', xbow[0]) # Running averages of the first batch of 8 tokens, each of size 2

x: torch.Size([4, 8, 2]) 

Batch [0]:
 tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]]) 

Running Averages:
 tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


**For each column, we have vertically averaged at each step from the first step until that current step.**
**For example, row one [ 0.1808, -0.0700] they are the same, but for row two [-0.3596, -0.9152] it will be the sum of *(row one + row two)/(sum([-0.3596, -0.9152]))* and so on ....**

**We can make this much more efficient using matrix multiplication and removing the `for` loops.**

<a id='202'></a>
## 2.2. Trick: Matrix Multiplication as Weighted Aggregation
-----
For each column, we have vertically averaged at each step from the first step until that current step by using `torch.tril` and matrix multiplication `a @ b`.

In [ ]:
torch.manual_seed(42)
a = torch.ones(3,3)     # try this as will : a = torch.randint(3,15,(3,3)).float()
b= torch.randint(0,10 ,(3,2)).float()
c = a @ b
print("a =")
print(a)
print("b =")
print(b)
print("c =")
print(c)

a =
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
b =
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c =
tensor([[14., 16.],
        [14., 16.],
        [14., 16.]])


**So now we can use torch.tril(torch.ones(3, 3)) and it will give as the lower traigular matrix of ones like the example below**


In [92]:
torch.tril(torch.ones(3,3))

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [ ]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))            # Lower triangular matrix of ones
a = a / a.sum(dim=1, keepdim=True)          # Normalize the matrix by dividing along each row  || dim=1 means use rows and dim=0 use columns

b = torch.randint(0, 10, (3, 2)).float()    # 3x2 matrix of random integers between 0 and 9
c = a @ b                                   # Matrix multiplication of a and b

print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])
